Confronto tra Embegging
1) utilizza la libreria 'Gensim' per caricare un modello Word2Vec pre-addestrato e calcola la somiglianza tra 'mela' e 'pesca'
2) Usa la libreria "Transformer" di Hugging Face per estrarre l'embedding della parola 'pesca' in due frasi diverse: "mi piace la pesca" e "ho mangiato una pesca dolce"
3) Calcola la distanza tra i due vettori e confronta con la rigidità del modello statico.

Suggerimento: osserva come la stessa parola produca vettori diversi nell'LLM.

In [1]:
import gensim.downloader as api

# Carichiamo un modello leggero (glove o word2vec)
# Scegliamo 'glove-wiki-gigaword-50' per la rapidità
model_static = api.load("glove-wiki-gigaword-50")

# Calcolo somiglianza (usando traduzioni o termini italiani se il modello lo permette)
# GloVe è rigido: 'mela' e 'pesca' avranno sempre lo stesso score.
sim = model_static.similarity('apple', 'peach') 
print(f"Somiglianza statica (GloVe) mela/pesca: {sim:.4f}")


import torch
from transformers import AutoTokenizer, AutoModel

# Usiamo un modello BERT italiano per coerenza
model_name = "dbmdz/bert-base-italian-xxl-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_context = AutoModel.from_pretrained(model_name)

frasi = [
    "Mi piace la pesca al lago", # Senso: Sport
    "Ho mangiato una pesca dolce" # Senso: Frutta
]

def get_word_vector(sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    outputs = model_context(**inputs)
    
    # Troviamo l'indice del token "pesca"
    token_logits = tokenizer.tokenize(sentence)
    word_idx = token_logits.index(word.lower()) + 1 # +1 per il token [CLS]
    
    # Estraiamo l'embedding dall'ultimo hidden state
    return outputs.last_hidden_state[0, word_idx, :]

vec_sport = get_word_vector(frasi[0], "pesca")
vec_frutta = get_word_vector(frasi[1], "pesca")



# Calcolo con PyTorch puro
cos = torch.nn.CosineSimilarity(dim=0)
distanza_contestuale = cos(vec_sport, vec_frutta)

print(f"Somiglianza tra i due contesti di 'pesca': {distanza_contestuale.item():.4f}")

# ANALISI
# GloVe: Distanza = 0 (perché il vettore è IDENTICO, la 'pesca' è una sola)
# BERT: La distanza riflette lo slittamento semantico.

Somiglianza statica (GloVe) mela/pesca: 0.5478


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2293.75it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-italian-xxl-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	

Somiglianza tra i due contesti di 'pesca': 0.5975
